# Consolidated Homing/Escape Detection Pipeline

Clean, streamlined workflow for Phase 1-3 homing detection analysis.  
Pipeline for learning features of manually labelled homings and applying them to a new homing detection logic

In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

from behave_analysis.process.process import Process
from behave_analysis.visualize.visualize_utils import open_tracking_data
from behave_analysis.analyze.behaviour.homings_escapes.homings import get_Homings
from behave_analysis.analyze.behaviour.homings_escapes.homing_curation_syd_viewer import homing_curation_syd_viewer, save_removed_runs

%matplotlib inline

In [ ]:
# Import experiments
from behave_analysis.database.Experiments.JAL004_ex import (
    JAL4_flip4_3Sept, JAL4_flip6_19Sept, JAL4_flip3_28aug, JAL4_flip5_11Sept
)
from behave_analysis.database.Experiments.JAL005_ex import JAL5_flip1_8Sept, JAL5_flip3_21Sept
from behave_analysis.database.Experiments.JAL006_ex import (
    JAL6_flip6_28mar, JAL6_flip4_21mar, JAL6_flip3_18mar, JAL6_flip5_25mar
)
from behave_analysis.database.Experiments.JAL007_ex import (
    JAL7_flip8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_flip9_16apr, JAL7_flip10_23apr
)
from behave_analysis.database.Experiments.JAL008_ex import (
    JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip5_14may, JAL8_flip4_10may
)

experiments_objects = {
    "JAL4_3rdSept": JAL4_flip4_3Sept,
    "JAL4_19thSept": JAL4_flip6_19Sept,
    "JAL4_28aug": JAL4_flip3_28aug,
    "JAL4_11thSept": JAL4_flip5_11Sept,
    "JAL5_8thSept": JAL5_flip1_8Sept,
    "JAL5_21stSept": JAL5_flip3_21Sept,
    "JAL6_28mar": JAL6_flip6_28mar,
    "JAL6_flip4_21mar": JAL6_flip4_21mar,
    "JAL6_flip3_18mar": JAL6_flip3_18mar,
    "JAL6_flip5_25mar": JAL6_flip5_25mar,
    "JAL7_sesh8_9apr": JAL7_flip8_9apr,
    "JAL7_flip5_22mar": JAL7_flip5_22mar,
    "JAL7_flip2_12mar": JAL7_flip2_12mar,
    "JAL7_sesh9_16apr": JAL7_flip9_16apr,
    "JAL7_23apr": JAL7_flip10_23apr,
    "JAL8_flip1_25apr": JAL8_flip1_25apr,
    "JAL8_flip2_29apr": JAL8_flip2_29apr,
    "JAL8_flip3_7may": JAL8_flip3_7may,
    "JAL8_14may": JAL8_flip5_14may,
    "JAL8_flip4_10may": JAL8_flip4_10may,
}

✓ Loaded 20 experiments


In [ ]:
settings = {"homings_speed_threshold": 4.0,  # cm/s, used to find bouts of running that may be homings
            "homings_gap_tolerance": 1,  # frames, used to merge bouts
            "homings_features_initial_window_s": 1.0,  # seconds, used to compute initial features of homings like acceleration and hdir change
            "homing_classification_target_recall": 0.9,  # minimum recall for a gate to be considered valid
            "homings_classification_recall_threshold": 0.9,  # minimum recall for a feature gate to be considered valid
            "homings_classification_precision_threshold": 0.1,  # minimum precision for a feature gate to be considered valid
            "homings_classification_auc_threshold": 0.9,  # or .8, minimum AUC for a feature gate to be considered valid
            "homings_classification_cohens_d_threshold": 1,  # minimum absolute Cohen's d for a feature gate to be considered valid
            "redo_compute": False,
            "homings_use_boris": False,
            "homings_distance_threshold": 25  # in cm, minimum length to be kept as a homings
            }

In [ ]:
# choose a session and load its data
e = 19

cond_list = ["shelter_only", "barrier_pre_flip", "barrier_post_flip"]
session_name = list(experiments_objects.keys())[e]
exp = experiments_objects[session_name]
session = Process(exp).load_session()

tracking_data = open_tracking_data(session)
video_df_path = os.path.join(session.base_path, session.processed_path, 'full_video_dataframe.csv')
video_df = pl.read_csv(video_df_path)

In [ ]:
# open database and check for run with matched settings - if it doesn't exist run it!
homings_dict = get_Homings(settings, session).get_homings(video_df=video_df, tracking_data=tracking_data)    

2026-06-17 14:48:13.776 | INFO     | behave_analysis.analyze.analyze_homing_consolidated:load_all_sessions:92 - Loading 20 sessions...
2026-06-17 14:48:13.887 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2026-06-17 14:48:42.860 | INFO     | behave_analysis.analyze.analyze_homing_consolidated:load_all_sessions:121 -   ✓ JAL4_3rdSept
2026-06-17 14:48:43.010 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2026-06-17 14:49:13.791 | INFO     | behave_analysis.analyze.analyze_homing_consolidated:load_all_sessions:121 -   ✓ JAL4_19thSept
2026-06-17 14:49:13.932 | INFO     | behave_analysis.process.process:load_session:117 - All data has been moved from winstor to ceph so we always load ceph data now
2026-06-17 14:49:40.784 | INFO     | behave_analysis.analyze.analyze_homing_consolidated:load_all_sessions:12

✓ Loaded 20 sessions


In [ ]:
# 4. load the homing results and open the syd viewer to manually curate
viewer, removed_runs = homing_curation_syd_viewer(homings_dict, session, tracking_data, video_df, settings)

In [ ]:
# 5. save the manual curation to the results dict and add a curated flag to the database entry